# Day 5 · Afternoon exercises (Exercise B)

**13:30–15:00 · then everyone back in the room at 15:00 for Peer review B**

Bayesian statistics for PhD researchers, Aalborg University 2026. Work in your group and keep your own copy of this notebook. Select **R (bayes)** and run the cells in order from a fresh kernel. Keep the folders `data/` and `models/` next to this notebook.

**If Stan does not compile on your laptop**, continue on UCloud or on a group member's laptop.

## Plan for this session

| Part | What | Guide time |
|---|---|---|
| B1 | Fit the regression and read it | 15 min |
| B2 | At 20 s: the mean versus one attempt | 10 min |
| B3 | A posterior predictive check: the spread at each target | 15 min |
| B4 | Revise the model and check again | 10 min |
| B5 | **Challenge, no template:** the whole class in one hierarchical regression | 25 min |
| — | Exit response and your group's question set | 15 min |

**Roles this block:** rotate them from the morning.

**The data.** The Day 1 time game without feedback: each person tried ten times at each of the targets **5, 10 and 15 seconds**, with eyes closed. If the lecturer has placed the class's data as `data/timing-class.csv`, the notebook uses it; otherwise it uses `data/timing-backup.csv`, **synthetic** data for 15 fictional people. The column `elapsed_seconds` is the recorded time. Today we model **one person**: yourself, or the first person in the data. Use the same person as this morning.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
library(cmdstanr)    # Stan from R
set.seed(2026)
options(width = 95, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")
theme_set(theme_minimal(base_size = 14) + theme(legend.position = "top"))
DATA <- "data"; MODELS <- "models"     # folders next to this notebook

In [ ]:
# The class's Day 1 data if the lecturer has put them here, otherwise the synthetic backup
file <- if (file.exists(file.path(DATA, "timing-class.csv"))) "timing-class.csv" else "timing-backup.csv"
file
d_all <- read_csv(file.path(DATA, file), show_col_types = FALSE) |>
  filter(feedback_condition == "no_feedback", !is.na(elapsed_seconds))
d_all |> count(participant, target_seconds) |> head()

me <- d_all$participant[1]          # or your own ID from Day 1, for example "G3P07"
d <- d_all |>
  filter(participant == me) |>
  mutate(x = target_seconds - 10)   # centred predictor: 0 at the 10-s target
me
nrow(d)
stan_data <- list(N = nrow(d), x = d$x, y = d$elapsed_seconds)

## B1 (15 min): fit the regression and read it

**The model** is the one from this morning, in `models/regression.stan` (*slides "The regression in Stan" and "The fit"*). On the target scale, the mean line has intercept α − 10β and slope β.

**Tasks.**

1. Fit the model (4 chains, `seed = 2026`) and check the diagnostics: R-hat, ESS, divergences (*Day 3 slide "Six numbers to check after every fit"*).
2. Report α, β and σ with 90% intervals, with units, and say in words what each means for this person.
3. Plot the data with 100 posterior mean lines and the line of perfect timing.
4. Compute P(β < 1 | y). What does it say about this person's clock?

In [ ]:
# Task 1 and 2
# mod <- cmdstan_model(file.path(MODELS, "regression.stan"))
# fit <- mod$sample(data = stan_data, chains = 4, seed = 2026, refresh = 0)
# fit$summary(c("alpha", "beta", "sigma"), "mean", ~quantile(.x, c(0.05, 0.95)), "rhat", "ess_bulk", "ess_tail")
# fit$diagnostic_summary()
# draws <- fit$draws(format = "df")

# Task 3
# some_draws <- draws |> slice_sample(n = 100)
# ggplot() +
#   geom_abline(data = some_draws, aes(intercept = alpha - 10 * beta, slope = beta), alpha = 0.12, colour = "tomato") +
#   ...

# Task 4: mean(draws$beta < 1)

**Your answer:** write a few sentences here.

## B2 (10 min): at 20 s, the mean versus one attempt

**The idea** (*slides "At 20 s: the mean versus one attempt" and "Prediction across all targets"*). In the Stan program, `mu_20 = alpha + beta * 10` is the person's **mean** recorded time at a 20-s target, and `y_20 ~ Normal(mu_20, sigma)` is **one attempt** there.

**Tasks.**

1. Give the 90% intervals of `mu_20` and `y_20`, and plot both histograms in one figure, with a legend.
2. Compute P(μ₂₀ < 20 | y) and P(ỹ₂₀ < 20 | y). Explain in words why they differ.
3. Nobody tried a 20-s target. What assumption are both predictions resting on?

In [ ]:
# Task 1
# quantile(draws$mu_20, c(0.05, 0.95));  quantile(draws$y_20, c(0.05, 0.95))
# ggplot(draws) +
#   geom_histogram(aes(x = y_20,  y = after_stat(density), fill = "one attempt at 20 s"), binwidth = 0.15, alpha = 0.6) +
#   geom_histogram(aes(x = mu_20, y = after_stat(density), fill = "mean at 20 s"),        binwidth = 0.08, alpha = 0.7) +
#   geom_vline(xintercept = 20, linetype = "dashed") +
#   scale_fill_manual(values = c("mean at 20 s" = "navy", "one attempt at 20 s" = "tomato"), name = NULL) +
#   labs(x = "recorded time (s)", y = "density", caption = "dashed line: the target, 20 s")

# Task 2: mean(draws$mu_20 < 20);  mean(draws$y_20 < 20)

**Your answer:** write a few sentences here.

## B3 (15 min): a posterior predictive check, the spread at each target

**The idea** (*slides "Posterior predictive checks", "A check: the spread at each target" and "The residuals show it too"*). If the model were right, would data like the person's be typical? For each posterior draw, simulate a replicated dataset at the same targets; compute a statistic T for each replicated dataset and for the real data, and compare. Here T is the **SD of the attempts at each target**: the model says the spread is the same σ at every target.

Run the helper cell first; it defines `replicated_sd()`, `ppc_plot()` and `ppc_table()`.

In [ ]:
# Helper for B3 and B4 (run it; you do not need to change it).
# For every posterior draw, replicate the person's attempts at the same targets,
# then take the SD of the replicated attempts at each target.
# This is what y_rep in the Stan program does; here we draw in R so that each
# replicated attempt keeps its target in the same row.
# growing = TRUE: the revised model, where the spread at target t is sigma * t / 10.
replicated_sd <- function(draws, growing = FALSE) {
  as_tibble(draws) |>
    select(.draw, alpha, beta, sigma) |>
    cross_join(d |> select(target_seconds, x)) |>
    mutate(spread = if (growing) sigma * target_seconds / 10 else sigma,
           y_rep  = rnorm(n(), alpha + beta * x, spread)) |>
    group_by(.draw, target_seconds) |>
    summarise(rep_sd = sd(y_rep), .groups = "drop")
}
observed_sd <- d |> group_by(target_seconds) |> summarise(obs_sd = sd(elapsed_seconds))

ppc_plot <- function(rep) {
  ggplot(rep, aes(x = rep_sd)) +
    geom_histogram(aes(fill = "replicated datasets"), bins = 40, alpha = 0.7) +
    geom_vline(data = observed_sd, aes(xintercept = obs_sd, colour = "observed data"), linewidth = 1.3) +
    facet_wrap(~ target_seconds, labeller = label_both, scales = "free_x") +
    scale_fill_manual(values = c("replicated datasets" = "grey60"), name = NULL) +
    scale_colour_manual(values = c("observed data" = "tomato"), name = NULL) +
    labs(x = "SD of the attempts at this target (s)", y = "number of draws")
}
ppc_table <- function(rep) {
  rep |> group_by(target_seconds) |>
    summarise(rep_q5 = quantile(rep_sd, 0.05), rep_q95 = quantile(rep_sd, 0.95)) |>
    left_join(observed_sd, by = "target_seconds")
}

**Tasks.**

1. Run the check for your fit: `rep_constant <- replicated_sd(draws)`, then `ppc_table(rep_constant)` and `ppc_plot(rep_constant)`. At which targets is the observed SD unusual among the replicated SDs?
2. Plot the residuals (recorded time − fitted mean, using the posterior means of α and β) against the target. What pattern do you see?
3. In one or two sentences: what does the model get wrong, and why could the summary of α, β and σ in B1 not tell you that?

In [ ]:
# Task 1
# rep_constant <- replicated_sd(draws)
# ppc_table(rep_constant)
# ppc_plot(rep_constant)

# Task 2
# res <- d |> mutate(fitted = mean(draws$alpha) + mean(draws$beta) * x,
#                    residual = elapsed_seconds - fitted)
# ggplot(res, aes(x = target_seconds, y = residual)) + geom_hline(yintercept = 0) + geom_jitter(width = 0.25, height = 0)

**Your answer:** write a few sentences here.

## B4 (10 min): revise the model and check again

**The revised model** (*slide "Revise: the spread grows with the target"*), in `models/regression_growing_sd.stan`: one changed line,

`y ~ normal(alpha + beta * x, sigma * (x + 10) / 10);`

so the spread at target t is σ · t / 10, and σ is the spread at a 10-s target.

**Tasks.**

1. Fit the revised model and check its diagnostics.
2. Run the same check with `replicated_sd(draws_growing, growing = TRUE)`. Does the observed SD now look typical at every target?
3. Compare the two models: β, and the 90% interval of one attempt at 20 s. Which prediction would you report, and why?

In [ ]:
# fit_growing <- cmdstan_model(file.path(MODELS, "regression_growing_sd.stan"))$sample(
#   data = stan_data, chains = 4, seed = 2026, refresh = 0)
# fit_growing$diagnostic_summary()
# draws_growing <- fit_growing$draws(format = "df")
# rep_growing <- replicated_sd(draws_growing, growing = TRUE)
# ppc_table(rep_growing);  ppc_plot(rep_growing)
# tibble(model = c("constant spread", "growing spread"),
#        beta = c(mean(draws$beta), mean(draws_growing$beta)),
#        y_20_q5  = c(quantile(draws$y_20, 0.05), quantile(draws_growing$y_20, 0.05)),
#        y_20_q95 = c(quantile(draws$y_20, 0.95), quantile(draws_growing$y_20, 0.95)))

**Your answer:** write a few sentences here.

## B5 (25 min): challenge, no template — the whole class in one hierarchical regression

*Slides "56 clocks", "A hierarchical regression", "The hierarchical regression in Stan", "Shrinkage of the slopes" and "At 20 s: a known person versus a new person".* So far you modelled one person. Now use **all 56 people's** attempts without feedback (`d_all`): every person j gets her own intercept α_j and slope β_j, drawn from a common population, and the spread grows with the target as in B4. There is no starter code: plan the steps in your group, then write them.

**Tasks.**

1. Fit the class model. Check the diagnostics. Report μ_β and τ_β with 90% intervals, and P(μ_β > 1 | y). What do they say about the class?
2. Compare your person's slope β_j with the β you found in B1, and find the three people whose slopes move most from their own least-squares slope. Why those?
3. Predict one attempt at a 20-s target for your person and for a new person from the same population. Compare your person's interval with B4.
4. The shared σ is inflated by a few people with erratic attempts. **Revise the model** so that each person has her own spread σ_j, with log σ_j ~ Normal(μ_logσ, τ_logσ) (non-centred). Fit it, list the four people with the largest σ_j, and redo task 3 for your person. Which prediction would you report?

<details><summary><b>Hints</b> (open only when you are stuck)</summary>

- **Stan program:** the slide "The hierarchical regression in Stan" shows the whole program; the demos folder has it as `demos/models/regression_hierarchical.stan`. Save your own copy in `models/` and fit it from there.
- **People as numbers:** Stan needs each person as an integer 1, …, J: `people <- unique(d_all$participant)` and `person = match(participant, people)` inside `mutate()`.
- **Data list:** `N`, `J`, `person`, `x` (target − 10) and `y`, all from `d_all`.
- **Your person's draws:** `j <- which(people == me)`, then the columns `` draws[[paste0("beta[", j, "]")]] `` and `` draws[[paste0("alpha[", j, "]")]] ``.
- **Own least-squares slope:** `group_by(participant) |> summarise(own_slope = cov(x, elapsed_seconds) / var(x))`.
- **One attempt at 20 s:** for each draw, `rnorm(S, alpha_j + beta_j * 10, sigma * 2)` (the spread at 20 s is twice the spread at 10 s). For a new person, draw α and β from the population first: `normal_rng(mu_alpha, tau_alpha)` in `generated quantities`.
- **σ_j per person:** declare `real mu_log_sigma; real<lower=0> tau_log_sigma; vector[J] z_sigma;`, then `vector[J] sigma = exp(mu_log_sigma + tau_log_sigma * z_sigma);` in `transformed parameters`, and in the likelihood use `sigma[person] .* (x + 10) / 10`. Priors such as `normal(0, 1)` for both are reasonable on the log scale.
- **Diagnostics:** if the bulk ESS of a parameter is below 400, run longer with `iter_sampling = 2000`.

</details>

In [ ]:
# B5: your code

**Your answer:** write a few sentences here.

**B6, optional** (*slides "An unusual attempt" and "Robust regression: Student-t errors"*): change one 15-s attempt to 24 s, as if the person forgot to click, and fit both `models/regression.stan` and `models/regression_t.stan` to the changed data. How much do β and σ move in each model? Start from `d_unusual <- d |> mutate(elapsed_seconds = if_else(target_seconds == 15 & trial == 1, 24, elapsed_seconds))`.

In [ ]:
# d_unusual <- d |> mutate(elapsed_seconds = if_else(target_seconds == 15 & trial == 1, 24, elapsed_seconds))
# data_unusual <- list(N = nrow(d_unusual), x = d_unusual$x, y = d_unusual$elapsed_seconds)

**Your answer:** write a few sentences here.

## Exit response

Write 80–120 words: What is a posterior predictive check? Describe the check you ran, what it showed, and how you changed the model in response.

**Your exit response:**

## Your group's question set

Question writers: write **three** questions about this block, each with your own worked solution:

1. a **concept** question (what does a result mean?),
2. a **computation** question (a number someone can check in R),
3. a **pitfall** question (a mistake that is easy to make).

The checker reviews the solutions. The reporter notes what your group found hardest, to ask in the plenary.

**Concept question:**

**Our solution:**

**Computation question:**

**Our solution:**

**Pitfall question:**

**Our solution:**

**Hardest open question (reporter):**

---
**Everyone back in the room at 15:00 for Peer review B.** Save this notebook (Ctrl/Cmd + S).